# Daily Electricity Consumption Analysis (Austria, 2016-2025)
This notebook performs ARMA modeling on the daily aggregated electricity consumption in Austria. We merge datasets from 2015 to 2025, truncate to start from 2016 to avoid artifacts, perform rigorous cleaning, and model seasonality using log-transformed harmonic regression.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings('ignore')

# Load both datasets
df1 = pd.read_csv('austria_hourly_load_2015_2019_MWh.csv')
df2 = pd.read_csv('austria_hourly_load_2020_2025_MWh.csv')

# Merge datasets
df = pd.concat([df1, df2], ignore_index=True)

# Pre-processing
df['timestamp_local'] = pd.to_datetime(df['timestamp_local'], utc=True)
df.set_index('timestamp_local', inplace=True)
df.sort_index(inplace=True)

## 1. Daily Aggregation (Raw)
# Truncate to start from 2016 to avoid earlier artifacts
df = df.loc['2016-01-01':]

# Aggregate to daily frequency
df_daily = df.resample('D').agg({'load_MWh': lambda x: x.sum(min_count=1)})

print(f"Total days in raw daily series (from 2016): {len(df_daily)}")
print(f"Range: {df_daily.index[0].date()} to {df_daily.index[-1].date()}")

## 2. Exploratory Data Analysis (EDA) on Daily Data
We analyze the raw daily aggregated series to identify artifacts and understand seasonal patterns.

In [ ]:
# 2.1 Raw Series Plot
plt.figure(figsize=(12, 4))
plt.plot(df_daily.index, df_daily['load_MWh'], linewidth=0.7)
plt.title('Raw Daily Electricity Load (Uncleaned)')
plt.ylabel('Load (MWh)')
plt.show()

# 2.2 Seasonality Analysis via Boxplots
eda_df = df_daily.copy()
eda_df['DayOfWeek'] = eda_df.index.dayofweek
eda_df['Month'] = eda_df.index.month

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.boxplot(x='DayOfWeek', y='load_MWh', data=eda_df, ax=axes[0])
axes[0].set_title('Load Distribution by Day of Week (0=Mon, 6=Sun)')
sns.boxplot(x='Month', y='load_MWh', data=eda_df, ax=axes[1])
axes[1].set_title('Load Distribution by Month')
plt.show()

# 2.3 Autocorrelation of Raw Data
fig, ax = plt.subplots(figsize=(12, 4))
plot_acf(df_daily['load_MWh'].dropna(), lags=40, ax=ax)
ax.set_title('ACF of Raw Daily Load (Scallop pattern shows weekly seasonality)')
plt.show()

## 3. Data Cleaning and Transformation
Based on the EDA, we remove outliers, interpolate gaps, and apply a log transformation.

In [ ]:
## 3.1 Advanced Cleaning
# - Physical Threshold: Austrian daily total load should realistically be at least ~120,000 MWh
df_daily.loc[df_daily['load_MWh'] < 120000, 'load_MWh'] = np.nan

# - Tight 3-Sigma Rolling Outlier Detection
rolling_mean = df_daily['load_MWh'].rolling(window=14, center=True).mean()
rolling_std = df_daily['load_MWh'].rolling(window=14, center=True).std()
outliers = (df_daily['load_MWh'] < rolling_mean - 3 * rolling_std) | (df_daily['load_MWh'] > rolling_mean + 3 * rolling_std)
df_daily.loc[outliers, 'load_MWh'] = np.nan

## 3.2 Interpolation and Log Transformation (Method 1.3)
df_daily['load_MWh'] = df_daily['load_MWh'].interpolate(method='linear')
df_daily.dropna(inplace=True)

# Apply log transformation to stabilize variance
df_daily['load_MWh_log'] = np.log(df_daily['load_MWh'])

print(f"Cleaned daily records: {len(df_daily)}")
plt.figure(figsize=(12, 4))
plt.plot(df_daily.index, df_daily['load_MWh_log'], linewidth=0.7, color='green')
plt.title('Cleaned Daily Electricity Consumption (Log Transformed)')
plt.show()

## 4. Train/Test Split
We hold out the last 30 days for testing.

In [ ]:
end_date = df_daily.index[-1]
test_start = end_date - pd.Timedelta(days=30)
train_end = test_start - pd.Timedelta(days=1)

train_data = df_daily.loc[:train_end]
test_data = df_daily.loc[test_start:end_date]

print(f"Training set: {len(train_data)} days")
print(f"Testing set:  {len(test_data)} days")

## 5. Removing Trend and Seasonality (Harmonic Regression)
With daily data, we model:
- **Trend:** Linear function of time.
- **Weekly Seasonality:** 7-day period (3 harmonics).
- **Annual Seasonality:** 365.25-day period (6 harmonics).

In [ ]:
t_train = np.arange(len(train_data))
X_trend = t_train.reshape(-1, 1)

def get_harmonics(t, period, n_harmonics):
    harmonics = []
    for i in range(1, n_harmonics + 1):
        harmonics.append(np.cos(2 * np.pi * i * t / period))
        harmonics.append(np.sin(2 * np.pi * i * t / period))
    return np.column_stack(harmonics)

X_weekly = get_harmonics(t_train, 7, 3)
X_annual = get_harmonics(t_train, 365.25, 6)

X_train = np.hstack((np.ones((len(t_train), 1)), X_trend, X_weekly, X_annual))
reg = LinearRegression(fit_intercept=False).fit(X_train, train_data['load_MWh_log'])

detrended_seasonal = reg.predict(X_train)
residuals = train_data['load_MWh_log'] - detrended_seasonal

plt.figure(figsize=(12, 4))
plt.plot(train_data.index[-365:], residuals.iloc[-365:])
plt.title('Residuals (Last 365 Days) after Removing Trend and Refined Seasonality (Log Scale)')
plt.show()

## 6. Stationarity and Best ARMA Selection
We verify stationarity and perform a grid search for the best ARMA model using AICC.

In [ ]:
adf_res = adfuller(residuals)
print(f"ADF Statistic: {adf_res[0]:.4f}, p-value: {adf_res[1]:.4g}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(residuals, ax=axes[0], lags=40)
plot_pacf(residuals, ax=axes[1], lags=40)
plt.show()

print("Selecting best ARMA model (AICC) using full training residuals...")
best_aicc = np.inf
best_order = (1, 0, 1)
aicc_results = []

for p in range(4):
    for q in range(4):
        if p == 0 and q == 0: continue
        try:
            model_fit = sm.tsa.ARIMA(residuals, order=(p, 0, q)).fit()
            aicc_results.append({'p': p, 'q': q, 'AICC': model_fit.aicc})
            if model_fit.aicc < best_aicc:
                best_aicc = model_fit.aicc
                best_order = (p, 0, q)
        except: continue

aicc_df = pd.DataFrame(aicc_results).sort_values(by='AICC')
print("\nARMA Model Comparison Table (Sorted by AICC):")
print(aicc_df.to_string(index=False))

print(f"\nBest order identified: {best_order}")
arma_model = sm.tsa.ARIMA(residuals, order=best_order).fit()
print(arma_model.summary())
arma_model.plot_diagnostics(figsize=(15, 10))
plt.show()

### Interpretation of Diagnostic Results
After fitting the best ARMA model, we examine the four residual diagnostic plots:
1. **Standardized Residuals:** We look for stationary white noise (random fluctuations around zero) with no obvious patterns.
2. **Histogram plus KDE:** The kernel density estimate (orange) should closely follow the N(0,1) distribution (green).
3. **Normal Q-Q:** Most points should lie on the 45-degree red line.
4. **Correlogram (ACF of Residuals):** Almost all autocorrelation spikes should fall within the blue significance bands.

## 7. Forecasting
We forecast the next 30 days of daily consumption, returning to the original scale.

In [ ]:
t_test = np.arange(len(train_data), len(train_data) + len(test_data))
X_test_weekly = get_harmonics(t_test, 7, 3)
X_test_annual = get_harmonics(t_test, 365.25, 6)
X_test = np.hstack((np.ones((len(t_test), 1)), t_test.reshape(-1, 1), X_test_weekly, X_test_annual))

deterministic_pred_log = reg.predict(X_test)
forecast_res = arma_model.get_forecast(steps=len(test_data))
arma_pred_log = forecast_res.predicted_mean
conf_int_log = forecast_res.conf_int(alpha=0.05)

# Exponentiate to return to original MWh scale
final_pred = np.exp(deterministic_pred_log + arma_pred_log.values)
lower = np.exp(deterministic_pred_log + conf_int_log.iloc[:, 0].values)
upper = np.exp(deterministic_pred_log + conf_int_log.iloc[:, 1].values)

plt.figure(figsize=(12, 6))
plt.plot(test_data.index, test_data['load_MWh'], label='Actual Daily Load', color='black')
plt.plot(test_data.index, final_pred, label='Forecast', color='red')
plt.fill_between(test_data.index, lower, upper, color='red', alpha=0.15, label='95% CI')
plt.title('Daily Electricity Consumption Forecast (30 Days)')
plt.ylabel('Daily Total Load (MWh)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

errors = test_data['load_MWh'].values - final_pred
mse = np.mean(errors**2)
rmse = np.sqrt(mse)
mape = np.mean(np.abs(errors / test_data['load_MWh'].values)) * 100

print(f"Forecast Performance Metrics (Daily):")
print(f"MSE:  {mse:.2f}")
print(f"RMSE: {rmse:.2f} MWh")
print(f"MAPE: {mape:.2f}%")